In [1]:
!pip install mediapipe opencv-python-headless
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [9]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import math
import numpy as np

In [18]:
LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]

In [13]:
def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

In [14]:
def calculate_ear_and_coords(eye_indices, landmarks, frame_w, frame_h):
    coords = []
    for idx in eye_indices:
        lm = landmarks[idx]
        coords.append((int(lm.x * frame_w), int(lm.y * frame_h)))

    v1 = euclidean_distance(coords[1], coords[5])
    v2 = euclidean_distance(coords[2], coords[4])
    h = euclidean_distance(coords[0], coords[3])

    ear = (v1 + v2) / (2.0 * h) if h != 0 else 0
    return ear, coords

In [15]:
def process_and_visualize_video(input_path, output_path, ear_threshold=0.15, consecutive_frames=1):
    # Initialize the Task API Options
    base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1
    )
    detector = vision.FaceLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error opening video: {input_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

    blink_count = 0
    eye_closed_frames = 0
    frame_count = 0

    print(f"Processing {input_path}... This might take a minute.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int(frame_count * 1000 / fps)
        frame_count += 1

        detection_result = detector.detect_for_video(mp_image, timestamp_ms)

        if detection_result.face_landmarks:
            face_landmarks = detection_result.face_landmarks[0]

            left_ear, left_coords = calculate_ear_and_coords(LEFT_EYE, face_landmarks, frame_w, frame_h)
            right_ear, right_coords = calculate_ear_and_coords(RIGHT_EYE, face_landmarks, frame_w, frame_h)

            avg_ear = (left_ear + right_ear) / 2.0

            left_eye_pts = np.array(left_coords, np.int32).reshape((-1, 1, 2))
            right_eye_pts = np.array(right_coords, np.int32).reshape((-1, 1, 2))

            # Draw resting eyes (Green)
            cv2.polylines(frame, [left_eye_pts], isClosed=True, color=(0, 255, 0), thickness=2)
            cv2.polylines(frame, [right_eye_pts], isClosed=True, color=(0, 255, 0), thickness=2)

            # Blink logic
            if avg_ear < ear_threshold:
                eye_closed_frames += 1
                # Draw closed eyes (Red)
                cv2.polylines(frame, [left_eye_pts], isClosed=True, color=(0, 0, 255), thickness=2)
                cv2.polylines(frame, [right_eye_pts], isClosed=True, color=(0, 0, 255), thickness=2)
            else:
                if eye_closed_frames >= consecutive_frames:
                    blink_count += 1
                eye_closed_frames = 0

            cv2.putText(frame, f"Blinks: {blink_count}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        out.write(frame)

    cap.release()
    out.release()
    detector.close()

    # Calculate the final statistics based on frames processed
    video_duration_sec = frame_count / fps if fps > 0 else 0
    blink_rate = blink_count / video_duration_sec if video_duration_sec > 0 else 0

    print(f"✅ Finished! Saved visually processed video to: {output_path}")
    print(f"--- Results for {input_path} ---")
    print(f"Total Blinks: {blink_count}")
    print(f"Video Duration: {video_duration_sec:.2f} seconds")
    print(f"Average Blink Rate: {blink_rate:.3f} blinks per second")
    print("-" * 30 + "\n")

    return blink_rate

In [16]:
movie_rate = process_and_visualize_video(
    input_path="/content/video_watching.mov",
    output_path="watching_movie_output.mp4",
    ear_threshold=0.15,
    consecutive_frames=1
)

Processing /content/video_watching.mov... This might take a minute.
✅ Finished! Saved visually processed video to: watching_movie_output.mp4
--- Results for /content/video_watching.mov ---
Total Blinks: 7
Video Duration: 69.47 seconds
Average Blink Rate: 0.101 blinks per second
------------------------------



In [17]:
reading_rate = process_and_visualize_video(
    input_path="/content/paper_reading.mov",
    output_path="reading_document_output.mp4",
    ear_threshold=0.15,
    consecutive_frames=1
)

Processing /content/paper_reading.mov... This might take a minute.
✅ Finished! Saved visually processed video to: reading_document_output.mp4
--- Results for /content/paper_reading.mov ---
Total Blinks: 22
Video Duration: 177.99 seconds
Average Blink Rate: 0.124 blinks per second
------------------------------

